# 📊 Phase 2: Price Monitoring & Violation Enforcement Pipeline
### Production-Grade Audit Pipeline for Minimum Advertised Price (MAP), Currency Normalization & Promotion Governance

---

## 🎯 Executive Overview & Business Objectives
In omni-channel e-commerce distribution, brand equity and channel partner trust are vulnerable to price erosion and unauthorized discounting. This production audit pipeline enforces **Minimum Advertised Price (MAP)** policies and **Lowest Possible Price (LPP)** constraints across five global markets (**IND, USA, UK, CAN, UAE**).

### 🌐 Key Architectural Enhancement: Cross-Border Currency Parity
- **Base Denomination in Price Master**: The master benchmark table (`price_list_table.csv`) sets `MAP` and `LPP` in **Indian Rupees (INR)** (e.g., LPP ranges from ₹5,000 to ₹100,000+).
- **Multi-Currency Marketplace Scraping**: Daily marketplace feeds in `seller_table.csv` record transactions in their respective local currencies:
  - `IND`: INR (₹)
  - `USA`: USD ($)
  - `CAN`: CAD ($)
  - `UK`: GBP (£)
  - `UAE`: AED (د.إ)
- **Currency Normalization**: To prevent false positives (where $460 USD would otherwise naively trigger a violation against ₹30,332 INR), this pipeline implements an **FX Normalization Engine** converting all advertised prices to base INR prior to benchmark evaluation, ensuring an **apple-to-apple parity audit**.

---

## 📋 Business Rules & Violation Classification Matrix
1. **`Price Below LPP` (Hard Floor Breach)**:
   - Evaluated when Advertised Price (normalized to base INR) falls strictly below the Lowest Possible Price floor:
     $$Advertised\_Price_{INR} < LPP$$
2. **`Unauthorized Discount (Non-Promotional SKU)`**:
   - Only pre-authorized SKUs in `promotional_skus.csv` are permitted to carry discounts below MAP.
   - Any non-promotional SKU whose advertised price falls below the MAP threshold is flagged:
     $$SKU \notin Whitelist \quad \text{AND} \quad Advertised\_Price_{INR} < MAP$$
3. **`Sub-Promotional Floor Violation`**:
   - Promotional SKUs whose advertised price falls below the calculated promotional floor.

---

## 📐 Data Dictionary & Target Schema
This notebook generates the final **`Price Monitoring Table`** as specified in the enterprise project charter:

| Column Name | Type | Description |
| :--- | :--- | :--- |
| **`SKU`** | `VARCHAR` | Unique Stock Keeping Unit identifier |
| **`Violation_date`** | `DATE` | Timestamp/Date when the price violation was observed |
| **`PL`** | `VARCHAR` | Product Line grouping code |
| **`Category`** | `VARCHAR` | Top-level product category (PC, Supply, Print Hardware) |
| **`Sub_category`** | `VARCHAR` | Specific product sub-tier (Laptop, Desktop, Ink, Laserjet, etc.) |
| **`Region`** | `VARCHAR` | Geographic market (IND, USA, UK, CAN, UAE) |
| **`Marketplace`** | `VARCHAR` | Platform where violation was observed (Amazon, Flipkart, etc.) |
| **`seller_name`** | `VARCHAR` | Marketplace display alias used by the vendor |
| **`homologated_name`** | `VARCHAR` | Canonical entity name resolved via fuzzy/homologation mapping |
| **`MAP_Price`** | `FLOAT` | Minimum Advertised Price floor mandated by brand |
| **`LPP`** | `FLOAT` | Lowest Possible Price hard constraint |
| **`Advertised_price`** | `FLOAT` | Observed scraped price in marketplace feed (local currency) |
| **`Season`** | `VARCHAR` | Active promotional season (`Q4`) |
| **`Promotional_price`**| `FLOAT` | Calculated compliant promotional price (for approved SKUs) |
| **`Violation_Type`** | `VARCHAR` | Precise audit classification for enforcement action |


### 1. Environment Setup & Configuration
We import standard data processing libraries, configure output formatting, and define the foreign exchange rate table (`FX_RATES`) for multi-regional currency normalization.

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Configure display options
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)

# Project paths setup
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
DATA_CODE_DIR = PROJECT_ROOT / 'data_code'
DB_TABLES_DIR = DATA_CODE_DIR / 'database_tables'
INTERMEDIATE_DIR = DATA_CODE_DIR / 'intermediate'

# Business configuration constants
ACTIVE_SEASON = 'Q4'

# Foreign Exchange (FX) Conversion Rates to Base Currency (INR)
# Resolves cross-border comparison against INR-denominated MAP and LPP benchmarks
FX_RATES = {
    'IND': 1.0,      # Base currency (INR)
    'USA': 83.50,    # USD to INR
    'CAN': 61.50,    # CAD to INR
    'UK':  105.00,   # GBP to INR
    'UAE': 22.70     # AED to INR
}

print(f"Project Root:      {PROJECT_ROOT}")
print(f"Database Tables:  {DB_TABLES_DIR}")
print(f"Intermediate Dir: {INTERMEDIATE_DIR}")
print(f"Active Season:    {ACTIVE_SEASON}")
print(f"FX Rates Table:   {FX_RATES}")


Project Root:      D:\Projects\Map Data Analysis Project
Database Tables:  D:\Projects\Map Data Analysis Project\data_code\database_tables
Intermediate Dir: D:\Projects\Map Data Analysis Project\data_code\intermediate
Active Season:    Q4
FX Rates Table:   {'IND': 1.0, 'USA': 83.5, 'CAN': 61.5, 'UK': 105.0, 'UAE': 22.7}


### 2. Ingesting Database Tables and Promotion Master Data
We load:
1. **`seller_table.csv`**: Daily scraped seller pricing feeds (147,900 observations across 5 global regions).
2. **`seller_mapping_table.csv`**: Entity resolution mapping table resolving marketplace aliases to homologated parent companies.
3. **`category_mapping_table.csv`**: Taxonomy hierarchy linking SKUs to PL, Category, and Sub-category.
4. **`price_list_table.csv`**: Core price list with MAP and LPP benchmarks.
5. **`promotional_skus.csv`**: White-list of SKUs authorized for promotional pricing.
6. **`promotion_condition.csv`**: Seasonal promotion business rules.


In [2]:
# Load relational tables
seller_df = pd.read_csv(DB_TABLES_DIR / 'seller_table.csv')
seller_map_df = pd.read_csv(DB_TABLES_DIR / 'seller_mapping_table.csv')
cat_map_df = pd.read_csv(DB_TABLES_DIR / 'category_mapping_table.csv')
price_list_df = pd.read_csv(DB_TABLES_DIR / 'price_list_table.csv')

# Load intermediate promotion governance files
promotional_skus_df = pd.read_csv(INTERMEDIATE_DIR / 'promotional_skus.csv')
prom_conditions_df = pd.read_csv(INTERMEDIATE_DIR / 'promotion_condition.csv')

# Convert authorized promotional SKUs into a fast lookup set
authorized_promotional_skus = set(promotional_skus_df['SKU'].dropna().unique())

print("=== Ingestion Summary ===")
print(f"- Seller Feed Rows:            {len(seller_df):,}")
print(f"- Seller Mappings:             {len(seller_map_df):,}")
print(f"- Category Mappings:           {len(cat_map_df):,}")
print(f"- Price List Entries:          {len(price_list_df):,}")
print(f"- Authorized Promotional SKUs: {len(authorized_promotional_skus):,}")


=== Ingestion Summary ===
- Seller Feed Rows:            147,900
- Seller Mappings:             900
- Category Mappings:           1,546
- Price List Entries:          1,546
- Authorized Promotional SKUs: 100


### 3. Reviewing Seasonal Promotion Rules
Let's inspect the active promotional condition matrix across categories for **Q4**:

In [3]:
print("Promotional Rules Matrix:")
display(prom_conditions_df[['CATEGORY', 'Q4']])


Promotional Rules Matrix:


,CATEGORY,Q4
0,SUP,20% off ink
1,PH,$10 off laser jet
2,PC,15 % off


### 4. Data Integration & Cross-Border Currency Normalization
We perform relational joins across the seller observations, seller mappings, category mappings, and price lists.

Next, we calculate **`Advertised_price_INR`** using our `FX_RATES` matrix. This ensures all comparisons with `MAP` and `LPP` are grounded in an identical base denomination (INR).

In [4]:
# Left join seller observations with homologated seller names
merged_df = seller_df.merge(
    seller_map_df,
    left_on='Seller_name',
    right_on='SELLER_NAME',
    how='left'
)

# Left join with category mapping hierarchy
merged_df = merged_df.merge(
    cat_map_df,
    on='SKU',
    how='left'
)

# Left join with price list benchmarks
merged_df = merged_df.merge(
    price_list_df[['SKU', 'MAP', 'LPP']],
    on='SKU',
    how='left'
)

# Filter to catalog-registered SKUs only
audit_df = merged_df.dropna(subset=['MAP', 'LPP']).copy()

# Currency Normalization Engine
audit_df['FX_Rate'] = audit_df['Region'].map(FX_RATES).fillna(1.0)
audit_df['Advertised_price_INR'] = np.round(audit_df['Advertised_price'] * audit_df['FX_Rate'], 2)

print(f"Total Merged Observations: {len(merged_df):,}")
print(f"Auditable Observations:    {len(audit_df):,}")
print("\nSample Normalized Prices across Regions:")
display(audit_df.groupby('Region')[['Advertised_price', 'FX_Rate', 'Advertised_price_INR', 'LPP']].head(1))


Total Merged Observations: 147,900
Auditable Observations:    143,550

Sample Normalized Prices across Regions:


,Advertised_price,FX_Rate,Advertised_price_INR,LPP
0,460.440,83.5,38446.74,30332.42
2,651.640,22.7,14792.23,12586.03
3,202.382,105.0,21250.11,17630.96
8,8630.000,1.0,8630.00,7537.45
10,134.240,61.5,8255.76,7537.45


### 5. Promotional Price Derivation Engine
According to the business charter and interview requirements:
1. **Authorization Gate**: Promotional prices apply **only** to SKUs listed in `promotional_skus.csv`.
2. **Category-Specific Q4 Rules**:
   - **PC**: $15\%$ discount applied directly to Advertised Price ($Advertised\_Price \times 0.85$).
   - **Supply (`SUP`)**: $20\%$ discount applied **only** if $Sub\_category == 'Ink'$.
   - **Print Hardware (`PH`)**: Fixed $10 off applied **only** if $Sub\_category == 'Laserjet'$. For the Indian market (`IND`), the $10 discount is converted to INR ($10 \times 83.50 = \text{₹}835$).
3. Non-eligible SKUs and categories retain `NaN` / null promotional pricing.


In [5]:
def calculate_promotional_price(row, promo_sku_set, fx_rates):
    """
    Computes the authorized promotional floor price for eligible SKUs based on Q4 rules in local currency.
    """
    sku = str(row['SKU']).strip()
    cat = str(row['CATEGORY']).strip()
    subcat = str(row['SUB_CATEGORY']).strip().lower()
    adv_price = row['Advertised_price']
    region = str(row['Region']).strip().upper()
    
    # Gate 1: Must be in authorized promotional SKUs whitelist
    if sku not in promo_sku_set:
        return np.nan
        
    # Rule 1: PC Category -> 15% discount
    if cat == 'PC':
        return round(adv_price * 0.85, 2)
        
    # Rule 2: Supply Category -> 20% discount strictly for Ink
    elif cat in ['SUP', 'Supply'] and subcat == 'ink':
        return round(adv_price * 0.80, 2)
        
    # Rule 3: Print Hardware -> $10 off strictly for Laserjet (converted to INR for IND)
    elif cat in ['PH', 'Print Hardware'] and 'laser' in subcat:
        usd_rate = fx_rates.get('USA', 83.50)
        discount_amount = 10.0 * usd_rate if region == 'IND' else 10.0
        return round(max(0.0, adv_price - discount_amount), 2)
        
    return np.nan

# Assign Season
audit_df['Season'] = ACTIVE_SEASON

# Apply promotional price calculation
audit_df['Promotional_price'] = audit_df.apply(
    lambda r: calculate_promotional_price(r, authorized_promotional_skus, FX_RATES),
    axis=1
)

promo_count = audit_df['Promotional_price'].notna().sum()
print(f"Calculated Promotional Prices: {promo_count:,} records ({promo_count/len(audit_df)*100:.2f}% of catalog)")
display(audit_df[audit_df['Promotional_price'].notna()][['SKU', 'CATEGORY', 'SUB_CATEGORY', 'Region', 'Advertised_price', 'Promotional_price']].head(5))


Calculated Promotional Prices: 7,335 records (5.11% of catalog)


,SKU,CATEGORY,SUB_CATEGORY,Region,Advertised_price,Promotional_price
12,81B19PA,PC,Laptop,UK,900.802,765.68
25,3YM22AA,SUP,Ink,USA,54.360,43.49
39,8L146PA,PC,Laptop,USA,3041.640,2585.39
70,381U0A,PH,Laserjet,USA,242.640,232.64
74,CN047AA,SUP,Ink,CAN,63.520,50.82


### 6. MAP & Promotional Violation Classification Logic
The compliance audit enforces three distinct violation checks using currency-normalized values:
1. **`Price Below LPP`**: Hard floor breach where $Advertised\_Price_{INR} < LPP$.
2. **`Unauthorized Discount (Non-Promotional SKU)`**: Where a seller discounts an unauthorized SKU below the MAP threshold ($Advertised\_Price_{INR} < MAP$).
3. **`Sub-Promotional Floor Violation`**: Where an authorized promotional SKU is advertised below its derived promotional floor ($Advertised\_Price < Promotional\_Price$).


In [6]:
def audit_violation_status(row, promo_sku_set):
    """
    Classifies violation severity and type based on currency-normalized pricing rules.
    """
    adv_inr = row['Advertised_price_INR']
    lpp = row['LPP']
    map_price = row['MAP']
    sku = str(row['SKU']).strip()
    adv_local = row['Advertised_price']
    promo_price = row['Promotional_price']
    is_promo = sku in promo_sku_set
    
    # Priority 1: Direct breach of Lowest Possible Price (LPP)
    if adv_inr < lpp:
        return 'Price Below LPP'
        
    # Priority 2: Non-promotional SKU carrying unauthorized discount below MAP
    elif not is_promo and adv_inr < map_price:
        return 'Unauthorized Discount (Non-Promotional SKU)'
        
    # Priority 3: Promotional SKU breaching promotional price floor
    elif is_promo and pd.notna(promo_price) and adv_local < promo_price:
        return 'Sub-Promotional Floor Violation'
        
    return np.nan

# Tag violation status
audit_df['Violation_Type'] = audit_df.apply(
    lambda r: audit_violation_status(r, authorized_promotional_skus),
    axis=1
)

# Extract violations only
violations_df = audit_df[audit_df['Violation_Type'].notna()].copy()

print("=== Violation Audit Results (With FX Currency Normalization) ===")
print(f"Total Observations Audited: {len(audit_df):,}")
print(f"Total Violations Detected:   {len(violations_df):,} ({len(violations_df)/len(audit_df)*100:.2f}%)")
print("\nBreakdown by Violation Type:")
print(violations_df['Violation_Type'].value_counts())
print("\nRegional Violation Distribution:")
display(violations_df.groupby('Region')['Violation_Type'].value_counts().unstack(fill_value=0))


=== Violation Audit Results (With FX Currency Normalization) ===
Total Observations Audited: 143,550
Total Violations Detected:   15,936 (11.10%)

Breakdown by Violation Type:
Violation_Type
Price Below LPP                                14283
Unauthorized Discount (Non-Promotional SKU)     1653
Name: count, dtype: int64

Regional Violation Distribution:


Violation_Type,Price Below LPP,Unauthorized Discount (Non-Promotional SKU)
Region,,
CAN,3061,313
IND,2703,329
UAE,3073,342
UK,2729,297
USA,2717,372


### 7. Aligning to Target Schema (Data Dictionary)
We standardize column names to strictly match the **`Price Monitoring Table`** specification in the project data dictionary:
- `Last time collected` $\rightarrow$ `Violation_date`
- `CATEGORY` $\rightarrow$ `Category`
- `SUB_CATEGORY` $\rightarrow$ `Sub_category`
- `Seller_name` $\rightarrow$ `seller_name`
- `HOMOLOGATED_NAME` $\rightarrow$ `homologated_name`
- `MAP` $\rightarrow$ `MAP_Price`


In [7]:
# Column renaming dictionary
schema_rename_map = {
    'Last time collected': 'Violation_date',
    'CATEGORY': 'Category',
    'SUB_CATEGORY': 'Sub_category',
    'Seller_name': 'seller_name',
    'HOMOLOGATED_NAME': 'homologated_name',
    'MAP': 'MAP_Price'
}

violations_df = violations_df.rename(columns=schema_rename_map)

# Exact target column sequence per Data Dictionary + Violation_Type
target_columns = [
    'SKU',
    'Violation_date',
    'PL',
    'Category',
    'Sub_category',
    'Region',
    'Marketplace',
    'seller_name',
    'homologated_name',
    'MAP_Price',
    'LPP',
    'Advertised_price',
    'Season',
    'Promotional_price',
    'Violation_Type'
]

price_monitoring_table = violations_df[target_columns].copy()

print("Price Monitoring Table Shape:", price_monitoring_table.shape)
print("\nColumns Schema:")
for idx, col in enumerate(price_monitoring_table.columns, 1):
    print(f"{idx:2d}. {col:<20} {str(price_monitoring_table[col].dtype):<10}")

display(price_monitoring_table.head(5))


Price Monitoring Table Shape: (15936, 15)

Columns Schema:
 1. SKU                  str       
 2. Violation_date       str       
 3. PL                   str       
 4. Category             str       
 5. Sub_category         str       
 6. Region               str       
 7. Marketplace          str       
 8. seller_name          str       
 9. homologated_name     str       
10. MAP_Price            float64   
11. LPP                  float64   
12. Advertised_price     float64   
13. Season               str       
14. Promotional_price    float64   
15. Violation_Type       str       


,SKU,Violation_date,PL,Category,Sub_category,Region,Marketplace,seller_name,homologated_name,MAP_Price,LPP,Advertised_price,Season,Promotional_price,Violation_Type
6,64X69AA,2025-12-21,HP64_PC_Monitor,PC,Monitor,UK,Amazon,E Basket-,Wegmans Food Market,13200.0,13041.60,74.448,Q4,NaN,Price Below LPP
11,3JA77AA,2025-12-21,HP3J_SUP_INK,SUP,Ink,IND,eBay,Avenue Overseas Private Limited,True Value Co.,2310.0,2159.85,2247.500,Q4,NaN,Unauthorized Discount (Non-Promotional SKU)
46,9D3M7PA,2025-12-21,HP9D_PC_Laptop,PC,Laptop,CAN,Flipkart,Mintree Premier Lifestyle,Defense Commissary Agency,42239.0,40802.87,452.800,Q4,NaN,Price Below LPP
64,54R50B,2025-12-21,HP54_PH_IJ,PH,Inkjet,UK,Amazon,Springtek Mattress,Michaels Stores,6369.0,6292.57,37.694,Q4,NaN,Price Below LPP
69,CZ134A,2025-12-21,HPCZ_SUP_INK,SUP,Ink,USA,Snapdeal,Mosaic Wellness Private Ltd.,Walmart,4384.0,4099.04,47.328,Q4,NaN,Price Below LPP


### 8. Compliance Intelligence & Offender Profiling
For stakeholder executive reporting and legal warning issuance:
1. **Top Offending Sellers (Homologated)**: Identifies parent accounts generating the highest volume of infractions.
2. **Marketplace Distribution**: Breakdown of violations across Amazon, Flipkart, etc.
3. **Temporal Trend Analysis**: Daily volume of violations over the observation period.


In [8]:
print("=== 1. Top 10 Offending Resellers (Homologated) ===")
top_sellers = price_monitoring_table['homologated_name'].value_counts().head(10)
display(top_sellers.to_frame('Violation Count'))

print("\n=== 2. Violations by Marketplace & Region ===")
market_breakdown = price_monitoring_table.groupby(['Marketplace', 'Region'])['SKU'].count().unstack(fill_value=0)
display(market_breakdown)

print("\n=== 3. Daily Violation Trend ===")
daily_trend = price_monitoring_table.groupby('Violation_date')['SKU'].count()
display(daily_trend.to_frame('Violations'))


=== 1. Top 10 Offending Resellers (Homologated) ===


,Violation Count
homologated_name,
Health Mart Systems,201
Sprouts Farmers Market,200
TJX Companies,199
Giant Eagle,195
Meijer,194
Wegmans Food Market,192
AVB Brandsource,192
True Value Co.,189
Dollar General,189



=== 2. Violations by Marketplace & Region ===


Region,CAN,IND,UAE,UK,USA
Marketplace,,,,,
Amazon,875,683,858,835,911
Flipkart,944,856,901,695,783
Snapdeal,569,736,894,720,622
eBay,986,757,762,776,773



=== 3. Daily Violation Trend ===


,Violations
Violation_date,
2025-12-21,731
2025-12-22,1414
2025-12-23,980
2025-12-24,931
2025-12-25,475
2025-12-26,1283
2025-12-27,778
2025-12-28,1174
2025-12-29,1131


### 9. Export to Database Tables
We export the sanitized dataset to `data_code/database_tables/price_monitoring_table.csv` and verify file persistence.


In [9]:
output_csv_path = DB_TABLES_DIR / 'price_monitoring_table.csv'
price_monitoring_table.to_csv(output_csv_path, index=False)

print(f" Successfully exported {len(price_monitoring_table):,} records to:")
print(f"-> {output_csv_path}")
print(f"File Size: {output_csv_path.stat().st_size / (1024 * 1024):.2f} MB")


 Successfully exported 15,936 records to:
-> D:\Projects\Map Data Analysis Project\data_code\database_tables\price_monitoring_table.csv
File Size: 2.02 MB
